In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv("customer_shopping_behavior.csv")

In [3]:
data.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [22]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   customer id             3900 non-null   int64   
 1   age                     3900 non-null   int64   
 2   gender                  3900 non-null   object  
 3   item purchased          3900 non-null   object  
 4   category                3900 non-null   object  
 5   purchase amount (usd)   3900 non-null   int64   
 6   location                3900 non-null   object  
 7   size                    3900 non-null   object  
 8   color                   3900 non-null   object  
 9   season                  3900 non-null   object  
 10  review rating           3900 non-null   float64 
 11  subscription status     3900 non-null   object  
 12  shipping type           3900 non-null   object  
 13  discount applied        3900 non-null   object  
 14  promo code used         

In [5]:
data.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [6]:
data.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

Here in data set there are 37 missing values , we have to handle them by replacing them 
WE are replacing them with category - wise rating 

In [3]:
#median category-wise 
data.groupby("Category")["Review Rating"].median()

Category
Accessories    3.8
Clothing       3.7
Footwear       3.8
Outerwear      3.8
Name: Review Rating, dtype: float64

In [4]:
#filling void sapces
data["Review Rating"]=data["Review Rating"].fillna(data.groupby("Category")["Review Rating"].transform("median"))

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3900 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

### Formating column names for simplification

In [16]:
data.columns= data.columns.str.lower()

In [23]:
data.columns=data.columns.str.replace(r" ","_")

In [24]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases', 'age_group'],
      dtype='object')

In [9]:
#creating age_group column 
labels=["young_adult","adult","middle_aged","senior"]
data["age_group"]=pd.qcut(data["age"],q=4,labels=labels)

In [10]:
data[["age","age_group"]].head()

,age,age_group
0,55,middle_aged
1,19,young_adult
2,50,middle_aged
3,21,young_adult
4,45,middle_aged


In [25]:
#creating column purchased frequency days 
frequency_mapping={
    "Fortnightly":14,
    "Weekly":7,
    "Annually":365,
    "Quarterly":90,
    "Monthly":30,
    'Bi-Weekly':14,
    'Every 3 Months':90,
}
data["purchases_frequency_days"]=data["frequency_of_purchases"].map(frequency_mapping)

In [26]:
data[["frequency_of_purchases","purchases_frequency_days"]].head()


,frequency_of_purchases,purchases_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365


In [27]:
#we have two columns "discount applied" and "promocode used ".
#Lets chck if they are identical 
(data["discount_applied"]==data["promo_code_used"]).all()

np.True_

In [28]:
# we are sure that if discount is applied then promocode is used so ..
del data["promo_code_used"]

In [29]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [30]:
# Database Configuration
DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": "rohan2005",
    "database": "analysis"        # Change this whenever needed
}

# Create Connection URL
url = URL.create(
    drivername="mysql+pymysql",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    database=DB_CONFIG["database"]
)

# Create Engine
engine = create_engine(url)

print("✅ Connected Successfully!")

✅ Connected Successfully!


In [32]:
data.to_sql("customer",engine,if_exists="replace",index=False)

3900

In [33]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   customer_id               3900 non-null   int64   
 1   age                       3900 non-null   int64   
 2   gender                    3900 non-null   object  
 3   item_purchased            3900 non-null   object  
 4   category                  3900 non-null   object  
 5   purchase_amount_(usd)     3900 non-null   int64   
 6   location                  3900 non-null   object  
 7   size                      3900 non-null   object  
 8   color                     3900 non-null   object  
 9   season                    3900 non-null   object  
 10  review_rating             3900 non-null   float64 
 11  subscription_status       3900 non-null   object  
 12  shipping_type             3900 non-null   object  
 13  discount_applied          3900 non-null   object